# 15: Census Demographics Integration

This notebook demonstrates **fetching demographic data from the Census Bureau API** and joining it with geographic boundaries for analysis and visualization.

## Key Capabilities

1. **CensusAPIClient** - Fetch ACS/Decennial demographic data
2. **Variable Groups** - Predefined sets of common Census variables
3. **Shape + Data Joins** - Combine TIGER boundaries with demographics
4. **Choropleth Visualization** - Map demographic data

## Prerequisites

```bash
pip install -e /path/to/siege_utilities[all]
```

**Optional but recommended:** Get a free Census API key at https://api.census.gov/data/key_signup.html

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

# Core imports
from siege_utilities.geo.census_api_client import (
    CensusAPIClient,
    get_demographics,
    get_population,
    get_income_data,
    get_education_data,
    get_housing_data,
    get_census_data_with_geometry,
    join_demographics_to_shapes,
    VARIABLE_GROUPS
)

from siege_utilities.geo.spatial_data import get_census_boundaries
from siege_utilities.geo.geoid_utils import construct_geoid, normalize_geoid

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

print("Imports successful!")

## 1. Understanding Variable Groups

The Census Bureau uses standardized variable codes (e.g., `B01001_001E` for total population).
siege_utilities provides predefined groups for common analyses.

In [ ]:
# List all predefined variable groups
print("Predefined Variable Groups")
print("=" * 60)

for group_name, variables in VARIABLE_GROUPS.items():
    print(f"\n{group_name}:")
    for var in variables[:5]:  # Show first 5 variables
        print(f"  - {var}")
    if len(variables) > 5:
        print(f"  ... and {len(variables) - 5} more")

## 2. Quick Demographic Fetches

Use convenience functions for common data needs.

In [ ]:
# Fetch population for all California counties
try:
    ca_pop = get_population(
        state='California',
        geography='county',
        year=2022
    )
    
    print(f"Retrieved population for {len(ca_pop)} California counties")
    print(f"\nColumns: {list(ca_pop.columns)}")
    
    # Rename for clarity and show top counties
    ca_pop = ca_pop.rename(columns={'B01001_001E': 'total_population'})
    print("\nTop 10 Most Populous California Counties:")
    display(ca_pop.nlargest(10, 'total_population')[['NAME', 'total_population']])
    
except Exception as e:
    print(f"Error: {e}")
    print("\nTip: Set CENSUS_API_KEY environment variable for reliable access")

In [ ]:
# Fetch income data for Texas counties
try:
    tx_income = get_income_data(
        state='TX',
        geography='county',
        year=2022
    )
    
    # Show median household income (B19013_001E)
    tx_income = tx_income.rename(columns={'B19013_001E': 'median_hh_income'})
    
    print("Texas Counties - Highest Median Household Income:")
    display(tx_income.nlargest(5, 'median_hh_income')[['NAME', 'median_hh_income']])
    
    print("\nTexas Counties - Lowest Median Household Income:")
    # Filter out null values before getting smallest
    valid = tx_income[tx_income['median_hh_income'] > 0]
    display(valid.nsmallest(5, 'median_hh_income')[['NAME', 'median_hh_income']])
    
except Exception as e:
    print(f"Error: {e}")

## 3. CensusAPIClient - Full Control

For more control, use the `CensusAPIClient` directly.

In [ ]:
# Initialize client (optionally with API key)
api_key = os.environ.get('CENSUS_API_KEY')  # None if not set
client = CensusAPIClient(api_key=api_key)

print(f"CensusAPIClient initialized")
print(f"API Key: {'Configured' if api_key else 'Not set (rate-limited)'}")
print(f"Cache directory: {client.cache_dir}")

In [ ]:
# Fetch specific variables with full control
try:
    ny_data = client.fetch_data(
        variables=['B01001_001E', 'B19013_001E', 'B25077_001E'],  # Pop, Income, Home Value
        year=2022,
        dataset='acs5',
        geography='county',
        state_fips='36',  # New York
        include_moe=True  # Include margin of error
    )
    
    print(f"Retrieved {len(ny_data)} New York counties")
    print(f"\nColumns: {list(ny_data.columns)}")
    
    # Show sample
    display(ny_data.head())
    
except Exception as e:
    print(f"Error: {e}")

## 4. Combined Geometry + Demographics

The most powerful feature: download TIGER boundaries AND demographics in one call,
automatically joined by GEOID.

In [ ]:
# Get California counties with median income - geometry included!
try:
    ca_income_gdf = get_census_data_with_geometry(
        year=2022,
        geography='county',
        state_fips='06',
        variables=['B19013_001E'],  # Median household income
        dataset='acs5'
    )
    
    print(f"Type: {type(ca_income_gdf)}")
    print(f"Rows: {len(ca_income_gdf)}")
    print(f"Has geometry: {ca_income_gdf.geometry is not None}")
    print(f"CRS: {ca_income_gdf.crs}")
    
except Exception as e:
    print(f"Error: {e}")
    # Fallback: manually join boundaries with demographics
    print("\nFallback: Creating demo with just boundaries...")
    ca_income_gdf = get_census_boundaries(2020, 'county', state_fips='06')
    ca_income_gdf = ca_income_gdf[ca_income_gdf['statefp'] == '06']
    ca_income_gdf['B19013_001E'] = [50000 + i * 1000 for i in range(len(ca_income_gdf))]

In [ ]:
# Create a choropleth map!
fig, ax = plt.subplots(1, 1, figsize=(12, 14))

ca_income_gdf.plot(
    column='B19013_001E',
    ax=ax,
    legend=True,
    legend_kwds={
        'label': 'Median Household Income ($)',
        'orientation': 'horizontal',
        'shrink': 0.8
    },
    cmap='YlGnBu',
    edgecolor='black',
    linewidth=0.3,
    missing_kwds={'color': 'lightgray', 'label': 'No Data'}
)

ax.set_title('California Counties\nMedian Household Income (2022 ACS 5-Year)', fontsize=14)
ax.axis('off')
plt.tight_layout()
plt.show()

## 5. Census Tract Level Analysis

For finer-grained analysis, use census tracts (smaller than counties).

In [ ]:
# Get tract-level data for Los Angeles County
try:
    la_tracts = get_census_data_with_geometry(
        year=2022,
        geography='tract',
        state_fips='06',
        county_fips='037',  # Los Angeles County
        variables=['B19013_001E'],
        dataset='acs5'
    )
    
    print(f"Los Angeles County has {len(la_tracts)} census tracts")
    
    # Visualize
    fig, ax = plt.subplots(1, 1, figsize=(14, 12))
    la_tracts.plot(
        column='B19013_001E',
        ax=ax,
        legend=True,
        cmap='RdYlGn',
        edgecolor='gray',
        linewidth=0.1,
        missing_kwds={'color': 'lightgray'}
    )
    ax.set_title('Los Angeles County Census Tracts\nMedian Household Income (2022)')
    ax.axis('off')
    plt.tight_layout()
    plt.show()
    
except Exception as e:
    print(f"Error (tract data requires API key): {e}")

## 6. Manual Join: Shapes + Demographics

If you already have boundaries, use `join_demographics_to_shapes()`.

In [ ]:
# Download boundaries separately
boundaries = get_census_boundaries(2020, 'county', state_fips='06')
boundaries = boundaries[boundaries['statefp'] == '06']
print(f"Downloaded {len(boundaries)} county boundaries")

# Fetch demographics separately
try:
    demographics = get_demographics(
        state='California',
        geography='county',
        year=2022
    )
    print(f"Fetched demographics for {len(demographics)} counties")
    
    # Join them
    joined = join_demographics_to_shapes(boundaries, demographics)
    print(f"\nJoined GeoDataFrame has {len(joined)} rows")
    print(f"Columns: {list(joined.columns)[:10]}...")
    
except Exception as e:
    print(f"Error: {e}")

## 7. Working with Different Datasets

The Census API provides multiple datasets:
- **acs5**: ACS 5-year estimates (most geographies, best coverage)
- **acs1**: ACS 1-year estimates (larger areas only)
- **dec/pl**: Decennial Census PL 94-171 (redistricting data)

In [ ]:
# Compare ACS 1-year vs 5-year for large metros
try:
    # ACS 5-year (2022)
    acs5 = client.fetch_data(
        variables=['B01001_001E'],
        year=2022,
        dataset='acs5',
        geography='county',
        state_fips='06'
    )
    print(f"ACS 5-year: {len(acs5)} counties")
    
    # ACS 1-year (2022) - only large counties
    acs1 = client.fetch_data(
        variables=['B01001_001E'],
        year=2022,
        dataset='acs1',
        geography='county',
        state_fips='06'
    )
    print(f"ACS 1-year: {len(acs1)} counties (only pop > 65k)")
    
except Exception as e:
    print(f"Error: {e}")

## Summary

### Quick Functions (No Client Needed)

| Function | Description |
|----------|-------------|
| `get_population()` | Total population counts |
| `get_demographics()` | Basic demographic variables |
| `get_income_data()` | Income-related variables |
| `get_education_data()` | Educational attainment |
| `get_housing_data()` | Housing characteristics |
| `get_census_data_with_geometry()` | Demographics + boundaries in one GeoDataFrame |

### CensusAPIClient Methods

| Method | Description |
|--------|-------------|
| `fetch_data()` | Fetch any variables with full control |
| `get_variable_metadata()` | Get info about a variable code |
| `list_available_variables()` | List variables in a dataset |
| `clear_cache()` | Clear cached data |

### Common Variable Codes

| Code | Description |
|------|-------------|
| B01001_001E | Total Population |
| B19013_001E | Median Household Income |
| B25077_001E | Median Home Value |
| B15003_022E | Bachelor's Degree |
| B17001_002E | Population Below Poverty |

### Geographic Levels

| Level | GEOID Length | Example |
|-------|--------------|----------|
| state | 2 | 06 (California) |
| county | 5 | 06037 (Los Angeles) |
| tract | 11 | 06037207400 |
| block group | 12 | 060372074001 |

### Resources

- [Census API Key Signup](https://api.census.gov/data/key_signup.html)
- [Census Variables Reference](https://api.census.gov/data.html)
- [ACS Subject Definitions](https://www.census.gov/programs-surveys/acs/technical-documentation/code-lists.html)